# Eksperyment badający metody na zbiorze make_friedman1

In [1]:
from sklearn.datasets import make_friedman1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [2]:
make_friedman1

<function sklearn.datasets._samples_generator.make_friedman1(n_samples=100, n_features=10, *, noise=0.0, random_state=None)>

In [2]:
X, y = make_friedman1(random_state=11)

In [3]:
X_df = pd.DataFrame(X)
y_df = pd.DataFrame(y)

In [4]:
df = pd.concat([X_df, y_df], axis=1)

Zmienne które są użyte do wyliczania y to 0,1,2,3,4

# Variance threshold

In [5]:
from sklearn.feature_selection import VarianceThreshold

#domyslny prog to 0 
sel = VarianceThreshold()
sel_var=sel.fit_transform(X)
sel_var = pd.DataFrame(sel_var)
sel_var


,0,1,2,3,4,5,6,7,8,9
0,0.180270,0.019475,0.463219,0.724934,0.420204,0.485427,0.012781,0.487372,0.941807,0.850795
1,0.729964,0.108736,0.893904,0.857154,0.165087,0.632334,0.020484,0.116737,0.316367,0.157912
2,0.758980,0.818275,0.344624,0.318799,0.111661,0.083953,0.712726,0.599543,0.055674,0.479797
3,0.401676,0.847979,0.717849,0.602064,0.552384,0.949102,0.986673,0.338054,0.239875,0.796436
4,0.063686,0.364616,0.070023,0.319368,0.070383,0.290264,0.790101,0.905400,0.792621,0.561819
...,...,...,...,...,...,...,...,...,...,...
95,0.142132,0.865395,0.114807,0.933345,0.929214,0.949463,0.779524,0.661688,0.388427,0.831868
96,0.580850,0.511924,0.135103,0.119053,0.674457,0.138835,0.702179,0.434966,0.128470,0.561244
97,0.790618,0.989851,0.247974,0.391557,0.484498,0.486974,0.235776,0.329805,0.089285,0.572236
98,0.996481,0.124271,0.720307,0.527561,0.327829,0.332847,0.986258,0.108496,0.588638,0.537075


Widzimy że Variance Threshold z domyślnym progiem równym 0 nie usnął nam żadnej zmiennej. Sprawdzimy teraz czy wyższe wartości progu pomogą nam wyrzucić niepotrzebne zmienne

In [7]:
X_df.columns.tolist()

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [6]:
threshold = [0.01,0.02,0.05,0.07,0.08,0.085]

for thr in threshold:
    sel = VarianceThreshold(thr)
    sel_var= sel.fit_transform(X)
    sel_var = pd.DataFrame(sel_var)
    selected_columns = X_df.columns[sel.get_support()]
    print("Wartość progu:", thr,"Pozostawione zmienne", list(selected_columns))

Wartość progu: 0.01 Pozostawione zmienne [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Wartość progu: 0.02 Pozostawione zmienne [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Wartość progu: 0.05 Pozostawione zmienne [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Wartość progu: 0.07 Pozostawione zmienne [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Wartość progu: 0.08 Pozostawione zmienne [0, 1, 4, 5, 7, 9]
Wartość progu: 0.085 Pozostawione zmienne [1, 5, 7]


Widzimy że od progu 0.08 zaczynamy usuwać zmienne, lecz z wiedzą jak powstaje nasz zbiór etykiet wiemy że akurat w przypadku tych danych ta metoda nie ma większego sensu i to że usuneliśmy niepotrzebne zmienne to przypadek

# Chi 2

Niestety w naszym przypadku metoda Chi-kwadrat nie zadziała bo nasz y jest ciągły a ta metoda działa tylko dla klasyfikacji binarnej

In [12]:
#from sklearn.feature_selection import SelectKBest, chi2
#chi_best = SelectKBest(chi2,k=5)
#chi_best.fit(X,y)

Możemy za to wykorzystać podobny test f_regression który bada korelacje Pearsona między każda cechą a y. Ten test powinien wykryć zmienne które są powiązane w y w sposób liniowy

# F_regression

In [13]:
from sklearn.feature_selection import SelectKBest, f_regression

f_reg_best = SelectKBest(f_regression,k=5)
f_reg_best.fit(X_df,y)
selected_columns = X_df.columns[f_reg_best.get_support()]
print(list(selected_columns))

[0, 1, 3, 4, 7]


In [12]:
# eksperyment dla różnych k

for k_par in range(1,6):
    f_reg_best = SelectKBest(f_regression,k=k_par)
    f_reg_best.fit(X_df,y)
    selected_columns = X_df.columns[f_reg_best.get_support()]
    print("Parametr k:",k_par," Wybrane kolumny: ", list(selected_columns))

Parametr k: 1  Wybrane kolumny:  [3]
Parametr k: 2  Wybrane kolumny:  [1, 3]
Parametr k: 3  Wybrane kolumny:  [0, 1, 3]
Parametr k: 4  Wybrane kolumny:  [0, 1, 3, 4]
Parametr k: 5  Wybrane kolumny:  [0, 1, 3, 4, 7]


Widzimy że ta metoda poradziła sobie już o wiele lepiej przy wyborze 5 najlepszych zmiennych wrzuca niestety zmienna 7 a nie 2 ale gdy zmiejszymy k do 4 to wszystkie otrzymane zmienne faktycznie wpływają na y

In [14]:
y_df

,0
0,9.487708
1,14.968015
2,13.514646
3,18.504056
4,7.972062
...,...
95,20.715721
96,15.266786
97,13.919618
98,11.678402


# F_classify

Ta metoda działa dla gdy zmienna celu jest zmienna klasyfikacyjną dlatego w naszym przypadku nie zadziała

# Information gain

In [6]:
#from sklearn.feature_selection import mutual_info_classif

Ta metoda też wymaga żeby y był zmienna dyskretną

In [9]:
from sklearn.feature_selection import mutual_info_regression

importance = mutual_info_regression(X,y,random_state=11)
feature_info = pd.Series(importance, X_df.columns).sort_values(ascending=False)
feature_info.head(5)

3    0.157281
9    0.141720
4    0.113725
0    0.087264
1    0.061559
dtype: float64

# Correlation Coefficient

In [21]:
y_df.columns = ["Target"]

In [25]:
df_train_transform = pd.concat([X_df.reset_index(drop=True), y_df.reset_index(drop=True)], axis=1)
corr_matrix = df_train_transform.corr()

target_corr = corr_matrix["Target"].abs().sort_values(ascending=False)
target_corr.head(6)

Target    1.000000
3         0.504075
1         0.351794
0         0.326819
4         0.208700
7         0.077689
Name: Target, dtype: float64

# Fisher score

Nie działa dla ciagłej zmiennej y

# Lasso

In [38]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import Lasso
 
lasso = SelectFromModel(Lasso(alpha=0.001))
lasso.fit(X, y)

SelectFromModel(estimator=Lasso(alpha=0.001))

In [39]:
lasso.estimator_.coef_

array([ 5.47807894,  6.39541674, -0.        ,  9.37124857,  4.30156207,
       -1.41539142, -0.02087793,  0.1925905 , -0.33672139, -1.01746808])

In [40]:
selected_features = X_df.columns[lasso.get_support()]
print(selected_features)

Index([0, 1, 3, 4, 5, 6, 7, 8, 9], dtype='int64')


In [43]:
alpha_values = np.logspace(-5, 3, 9)

for a in alpha_values:
    lasso = SelectFromModel(Lasso(alpha=a, max_iter=5000, random_state=11))
    lasso.fit(X,y)
    selected_features = X_df.columns[lasso.get_support()]
    print("Alfa : ", a, "Wybrane zmienne: ", list(selected_features))

Alfa :  1e-05 Wybrane zmienne:  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Alfa :  0.0001 Wybrane zmienne:  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Alfa :  0.001 Wybrane zmienne:  [0, 1, 3, 4, 5, 6, 7, 8, 9]
Alfa :  0.01 Wybrane zmienne:  [0, 1, 3, 4, 5, 7, 8, 9]
Alfa :  0.1 Wybrane zmienne:  [0, 1, 3, 4]
Alfa :  1.0 Wybrane zmienne:  []
Alfa :  10.0 Wybrane zmienne:  []
Alfa :  100.0 Wybrane zmienne:  []
Alfa :  1000.0 Wybrane zmienne:  []


In [56]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=3,random_state=11)
model.fit(X,y)

importances = model.feature_importances_


feature_importance_df = pd.DataFrame({
    "feature": X_df.columns,
    "importance": importances
})
feature_importance_df = feature_importance_df.sort_values(by="importance", ascending=False).reset_index()
print(feature_importance_df[["feature","importance"]])

   feature  importance
0        3    0.315003
1        1    0.288004
2        0    0.177794
3        5    0.043880
4        4    0.042928
5        2    0.041004
6        9    0.039686
7        7    0.027643
8        8    0.012827
9        6    0.011231


# Metody oparte na wrapperach

Wszystkie te metody bedziemy sprawdzam dla bazowego modelu regresji liniowej

In [16]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

# Forward Feature Selection

In [17]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

sfs_forward = SFS(
    model,
    k_features=5,      # wybieramy 5 cech
    forward=True,      # forward = dodawanie cech sekwencyjnie
    floating=False,    # bez floating
    scoring='r2',      # metryka regresji
    cv=5               # cross-validation
)

In [22]:
%time sfs_forward.fit(X_df, y_df)

CPU times: total: 359 ms
Wall time: 377 ms


SequentialFeatureSelector(estimator=LinearRegression(), k_features=(5, 5),
                          scoring='r2')

In [24]:
selected_columns = [X_df.columns[i] for i in sfs_forward.k_feature_idx_]
print("Wybrane cechy:", selected_columns)

print("Kolejność dodawania:", sfs_forward.k_feature_names_)

Wybrane cechy: [0, 1, 3, 4, 5]
Kolejność dodawania: (0, 1, 3, 4, 5)


# Backward Feature Elimination

In [25]:
sfs_backward = SFS(
    model,
    k_features=5,      # wybieramy 5 cech
    forward=False,      
    floating=False,    # bez floating
    scoring='r2',      # metryka regresji
    cv=5               # cross-validation
)

In [26]:
%time sfs_backward.fit(X_df, y_df)

CPU times: total: 422 ms
Wall time: 458 ms


SequentialFeatureSelector(estimator=LinearRegression(), forward=False,
                          k_features=(5, 5), scoring='r2')

In [27]:
selected_columns = [X_df.columns[i] for i in sfs_forward.k_feature_idx_]
print("Wybrane cechy:", selected_columns)

print("Kolejność dodawania:", sfs_forward.k_feature_names_)

Wybrane cechy: [0, 1, 3, 4, 5]
Kolejność dodawania: (0, 1, 3, 4, 5)


In [30]:
print("Indeksy wybranych cech:", sfs_backward.k_feature_idx_)
selected_columns = [X_df.columns[i] for i in sfs_backward.k_feature_idx_]
print("Wybrane cechy:", selected_columns)
print("Kolejność usuwania/dodawania:", sfs_backward.k_feature_names_)

Indeksy wybranych cech: (0, 1, 3, 4, 5)
Wybrane cechy: [0, 1, 3, 4, 5]
Kolejność usuwania/dodawania: (0, 1, 3, 4, 5)


In [32]:
metrics = sfs_backward.get_metric_dict()
for k in metrics:
    print(f"Iteracja {k}: cechy = {metrics[k]['feature_idx']}, score = {metrics[k]['avg_score']:.4f}")

Iteracja 10: cechy = (0, 1, 2, 3, 4, 5, 6, 7, 8, 9), score = 0.5456
Iteracja 9: cechy = (0, 1, 3, 4, 5, 6, 7, 8, 9), score = 0.5537
Iteracja 8: cechy = (0, 1, 3, 4, 5, 6, 7, 9), score = 0.5603
Iteracja 7: cechy = (0, 1, 3, 4, 5, 6, 9), score = 0.5659
Iteracja 6: cechy = (0, 1, 3, 4, 5, 9), score = 0.5662
Iteracja 5: cechy = (0, 1, 3, 4, 5), score = 0.5661


In [33]:
removed_features = []
all_features = set(range(X.shape[1]))

for k in sorted(metrics.keys()):
    remaining = set(metrics[k]['feature_idx'])
    removed = all_features - remaining
    removed_features.append(removed)
    all_features = remaining

print("Kolejne cechy usuwane:", removed_features)

Kolejne cechy usuwane: [{2, 6, 7, 8, 9}, set(), set(), set(), set(), set()]
